# Phase 2

## 1. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import statsmodels.formula.api as smf

DATA_ROOT = Path("/Users/theocoronges/Documents/internship2026/data")
OD_DIR  = DATA_ROOT / "od"
GEO_DIR = DATA_ROOT / "geometries"

ZONES, VERSION = "distritos", 2
WEEKS = [
    ("2022-10-17", "2022-10-23"),
    ("2023-10-16", "2023-10-22"),
    ("2024-10-14", "2024-10-20"),
    ("2025-10-13", "2025-10-19"),
]

# pop lookup from district shapes (population is text -> coerce)
gdf = gpd.read_file(GEO_DIR / f"{ZONES}_{VERSION}.geojson")
pop = pd.to_numeric(gdf.set_index("id")["population"], errors="coerce")

def load_week(week_num, mode):
    week = f"{WEEKS[week_num][0]}_{WEEKS[week_num][1]}"
    return pd.read_parquet(f"{OD_DIR}/{week}/Viajes_distritos_{week}_v2_{mode}.parquet")

wk1 = load_week(3, "activity")          # 2025 week
print(wk1.shape[0], "rows |", gdf.shape[0], "districts")

46953143 rows | 3792 districts


## 2. Helpers

In [ ]:
def add_is_weekend(df):
    df = df.copy()
    # sat/sun -> 1, else 0
    df["is_weekend"] = np.where(pd.to_datetime(df["date"]).dt.weekday.isin([5, 6]), 1, 0)
    return df


def to_pairs(df, slice_col):
    # collapse to one row per slice+origin+dest, sum trips & km
    pairs = (df.groupby([slice_col, "id_origin", "id_destination"], as_index=False)
               [["n_trips", "trips_total_length_km"]].sum())

    # avg trip length = total km / total trips (ratio, no per-day adj)
    pairs["avg_distance"] = pairs["trips_total_length_km"] / pairs["n_trips"]

    # trips per day so slices w/ diff #days compare (/2 wknd, /5 wkday, /7 week)
    n_days = df.groupby(slice_col)["date"].nunique()
    pairs["n_trips"] = pairs["n_trips"] / pairs[slice_col].map(n_days)
    return pairs


def add_pop_logs(pairs):
    pairs = pairs.copy()
    pairs["Pi"] = pairs["id_origin"].map(pop)        # origin pop
    pairs["Pj"] = pairs["id_destination"].map(pop)   # dest pop

    # drop no-pop zones (foreign) + non-positive (log needs >0)
    pairs = pairs.dropna(subset=["Pi", "Pj"])
    pairs = pairs[(pairs["n_trips"] > 0) & (pairs["avg_distance"] > 0)]

    # logs
    pairs["log_trips"] = np.log(pairs["n_trips"])
    pairs["log_Pi"] = np.log(pairs["Pi"])
    pairs["log_Pj"] = np.log(pairs["Pj"])
    pairs["log_dij"] = np.log(pairs["avg_distance"])
    return pairs


def fit_gravity(pairs):
    return smf.ols("log_trips ~ log_Pi + log_Pj + log_dij", data=pairs).fit()


def coefs(m):
    # coefs + fit quality
    return {"n": int(m.nobs),
            "intercept": m.params["Intercept"],
            "beta_Pi": m.params["log_Pi"],     # origin pop elasticity
            "beta_Pj": m.params["log_Pj"],     # dest pop elasticity
            "gamma": -m.params["log_dij"],     # distance decay (flip sign)
            "R2": m.rsquared}

## 3. Weekday vs weekend

In [5]:
dt_pairs = add_pop_logs(to_pairs(add_is_weekend(wk1), "is_weekend"))

models_dt = {}
for val, label in [(0, "weekday"), (1, "weekend")]:
    m = fit_gravity(dt_pairs[dt_pairs["is_weekend"] == val])
    models_dt[label] = m
    print("=" * 70)
    print(label.upper())
    print(m.summary())

WEEKDAY
                            OLS Regression Results                            
Dep. Variable:              log_trips   R-squared:                       0.612
Model:                            OLS   Adj. R-squared:                  0.612
Method:                 Least Squares   F-statistic:                 5.508e+05
Date:                Sun, 28 Jun 2026   Prob (F-statistic):               0.00
Time:                        22:45:24   Log-Likelihood:            -1.6691e+06
No. Observations:             1048133   AIC:                         3.338e+06
Df Residuals:                 1048129   BIC:                         3.338e+06
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.4864      0.015    -32.426 

In [6]:
# side by side
pd.DataFrame({k: coefs(v) for k, v in models_dt.items()}).T

,n,intercept,beta_Pi,beta_Pj,gamma,R2
weekday,1048133.0,-0.486365,0.419743,0.416786,1.292984,0.611868
weekend,720385.0,0.106642,0.357921,0.388334,1.142138,0.571504


## 4. By trip purpose

In [7]:
act_pairs = add_pop_logs(to_pairs(wk1, "activity_destination"))

models_act = {}
for act in sorted(act_pairs["activity_destination"].unique()):
    m = fit_gravity(act_pairs[act_pairs["activity_destination"] == act])
    models_act[act] = m
    print("=" * 70)
    print(act)
    print(m.summary())

home
                            OLS Regression Results                            
Dep. Variable:              log_trips   R-squared:                       0.579
Model:                            OLS   Adj. R-squared:                  0.579
Method:                 Least Squares   F-statistic:                 3.341e+05
Date:                Sun, 28 Jun 2026   Prob (F-statistic):               0.00
Time:                        22:51:03   Log-Likelihood:            -1.1646e+06
No. Observations:              727668   AIC:                         2.329e+06
Df Residuals:                  727664   BIC:                         2.329e+06
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -1.0239      0.018    -56.627    

In [ ]:
pd.DataFrame({k: coefs(v) for k, v in models_act.items()}).T